In [2]:
################################# TOPIC 5 - GROUP BY AND AGGREGATION #########################################

In [3]:
# Groupby and aggregation - Summarizing metrics per server, per priority, per time window

In [4]:
# LOAD DATASET TO WORK ON
import pandas as pd
import numpy as np

# ── Load all three datasets ────────────────────────────────────────────
df_metrics = pd.read_csv(
    "server_metrics.csv",
    parse_dates=["timestamp"],
    dtype={"server_id": "category"},
    na_values=["N/A", "null", "—"]
)

df_tickets = pd.read_csv(
    "incidents.csv",
    parse_dates=["created_at", "resolved_at"]
)

df_logs = pd.read_csv(
    "app_logs.csv",
    parse_dates=["timestamp"]
)

# ── Confirm shapes ─────────────────────────────────────────────────────
print(f"df_metrics : {df_metrics.shape}")
print(f"df_tickets : {df_tickets.shape}")
print(f"df_logs    : {df_logs.shape}")

df_metrics : (525, 7)
df_tickets : (213, 7)
df_logs    : (315, 5)


In [5]:
# Basic groupby + single aggregation
df_metrics.groupby("server_id", observed=True)["cpu_pct"].mean()

server_id
srv-01    59.926667
srv-02    59.931429
srv-03    58.565714
srv-04    59.409524
srv-05    62.332381
Name: cpu_pct, dtype: float64

In [6]:
# Multiple aggregations - agg()
# on one column
df_metrics.groupby("server_id", observed=True)["cpu_pct"].agg(["mean", "max", "min", "std"]).round(2)

,mean,max,min,std
server_id,,,,
srv-01,59.93,98.4,21.8,24.37
srv-02,59.93,98.2,20.7,23.06
srv-03,58.57,98.5,20.9,23.10
srv-04,59.41,98.6,20.6,25.00
srv-05,62.33,96.7,20.6,22.01


In [7]:
# Multiple stats on one column
df_metrics.groupby("server_id", observed=True)["cpu_pct"].agg(["mean", "max", "min", "std"]).round(2)

,mean,max,min,std
server_id,,,,
srv-01,59.93,98.4,21.8,24.37
srv-02,59.93,98.2,20.7,23.06
srv-03,58.57,98.5,20.9,23.10
srv-04,59.41,98.6,20.6,25.00
srv-05,62.33,96.7,20.6,22.01


In [8]:
# Naming each aggregation explicitly
df_metrics.groupby("server_id", observed=True).agg(
    avg_cpu = ("cpu_pct", "mean"),
    max_cpu = ("cpu_pct", "max"),
    avg_memory = ("memory_pct", "mean"),
    avg_response = ("response_ms", "mean"),
    total_records = ("cpu_pct", "count")
).round(2)

,avg_cpu,max_cpu,avg_memory,avg_response,total_records
server_id,,,,,
srv-01,59.93,98.4,62.51,643.57,105
srv-02,59.93,98.2,64.46,633.02,105
srv-03,58.57,98.5,65.09,622.77,105
srv-04,59.41,98.6,62.21,632.40,105
srv-05,62.33,96.7,62.10,590.14,105


In [9]:
# Groupby on multiple columns
# Average cpu per server per status
df_metrics.groupby(["server_id", "status"], observed=True)["cpu_pct"].mean().round(2)

server_id  status  
srv-01     critical    58.96
           ok          59.29
           warning     62.57
srv-02     critical    61.67
           ok          57.60
           warning     73.05
srv-03     critical    55.89
           ok          57.33
           warning     68.10
srv-04     critical    62.85
           ok          60.21
           warning     56.25
srv-05     critical    50.13
           ok          61.36
           warning     70.87
Name: cpu_pct, dtype: float64

In [10]:
# Groupby on incidents - ticket analysis
# Count tickets per category
df_tickets.groupby("category")["ticket_id"].count().sort_values(ascending=False)

category
CPU High        50
Disk Full       46
Service Down    46
Memory Leak     36
Network Lag     35
Name: ticket_id, dtype: int64

In [11]:
# Resolution rate per category
df_tickets.groupby("category").agg(
    total = ("ticket_id", "count"),
    resolved = ("resolved", "sum"),
).assign(
    resolution_rate = lambda x:(x["resolved"] / x["total"] * 100 ).round(1)
).sort_values("resolution_rate")

,total,resolved,resolution_rate
category,,,
Network Lag,35,19,54.3
Service Down,46,25,54.3
Disk Full,46,26,56.5
Memory Leak,36,22,61.1
CPU High,50,33,66.0


In [12]:
# Groupby on logs - error rate analysis
# log count per server per log level
df_logs.groupby(["server_id", "log_level"])["message"].count().unstack(fill_value=0)

log_level,CRITICAL,ERROR,INFO,WARNING
server_id,,,,
srv-01,7,5,26,7
srv-02,8,14,32,18
srv-03,6,11,30,19
srv-04,10,6,27,20
srv-05,10,11,34,14


In [13]:
# Error rate per server - ERROR + CRITICAL as % of total logs
log_counts = df_logs.groupby("server_id")["log_level"].value_counts(normalize=True).mul(100).round(1).unstack(fill_value=0)
log_counts["error_rate"] = log_counts.get("ERROR", 0) + log_counts.get("CRITICAL", 0)
log_counts[["ERROR", "CRITICAL", "error_rate"]].sort_values("error_rate", ascending=False)

log_level,ERROR,CRITICAL,error_rate
server_id,,,
srv-02,19.4,11.1,30.5
srv-05,15.9,14.5,30.4
srv-01,11.1,15.6,26.7
srv-03,16.7,9.1,25.8
srv-04,9.5,15.9,25.4


In [14]:
# transform()
# Add fleet average cpu as a column - for comparison per row 
df_metrics["fleet_avg_cpu"] = df_metrics.groupby("server_id", observed=True)["cpu_pct"].transform("mean")
print(df_metrics["fleet_avg_cpu"])

0      59.926667
1      59.931429
2      58.565714
3      59.409524
4      62.332381
         ...    
520    59.926667
521    59.931429
522    58.565714
523    59.409524
524    62.332381
Name: fleet_avg_cpu, Length: 525, dtype: float64


In [15]:
# How far is each reading from its server's average ? 
df_metrics["cpu_deviation"] = df_metrics["cpu_pct"] - df_metrics["fleet_avg_cpu"]
print(df_metrics[["server_id", "cpu_pct", "fleet_avg_cpu", "cpu_deviation"]].head(10))

  server_id  cpu_pct  fleet_avg_cpu  cpu_deviation
0    srv-01     49.6      59.926667     -10.326667
1    srv-02     32.3      59.931429     -27.631429
2    srv-03     21.6      58.565714     -36.965714
3    srv-04     34.5      59.409524     -24.909524
4    srv-05     68.3      62.332381       5.967619
5    srv-01     82.0      59.926667      22.073333
6    srv-02     68.0      59.931429       8.068571
7    srv-03     83.9      58.565714      25.334286
8    srv-04     29.6      59.409524     -29.809524
9    srv-05     72.3      62.332381       9.967619


In [16]:
# filter() - keep only groups meeting a condition

#keep only servers where average CPU > 60
high_load_servers = df_metrics.groupby("server_id", observed=True).filter(lambda x: x["cpu_pct"].mean() > 60)
print(high_load_servers["server_id"].unique())

['srv-05']
Categories (5, object): ['srv-01', 'srv-02', 'srv-03', 'srv-04', 'srv-05']


In [17]:
# resample() preview - hourly aggregation

# Hourly average CPU across entire fleet
df_metrics.set_index("timestamp").resample("1h")["cpu_pct"].mean().head()

timestamp
2026-01-01 00:00:00    41.26
2026-01-01 01:00:00    67.16
2026-01-01 02:00:00    76.80
2026-01-01 03:00:00    59.98
2026-01-01 04:00:00    55.54
Freq: h, Name: cpu_pct, dtype: float64

In [18]:
# Lab — GroupBy + Aggregation
# Task 1 — server_metrics
# For each server compute: mean cpu, max cpu, mean memory, max response_ms, 
# total row count. Use named aggregations. Sort by mean cpu descending. 
# Which server is most loaded on average?

In [19]:
df_metrics.head(5)

,timestamp,server_id,cpu_pct,memory_pct,response_ms,disk_pct,status,fleet_avg_cpu,cpu_deviation
0,2026-01-01,srv-01,49.6,94.6,891.8,68.9,ok,59.926667,-10.326667
1,2026-01-01,srv-02,32.3,33.9,1046.1,69.1,ok,59.931429,-27.631429
2,2026-01-01,srv-03,21.6,96.0,1007.3,43.8,ok,58.565714,-36.965714
3,2026-01-01,srv-04,34.5,50.7,653.5,58.1,ok,59.409524,-24.909524
4,2026-01-01,srv-05,68.3,39.5,386.0,53.8,ok,62.332381,5.967619


In [20]:
#df_metrics.columns.tolist()
df_metrics.groupby("server_id", observed=True).agg(
    mean_cpu = ("cpu_pct", "mean"),
    max_cpu = ("cpu_pct", "max"),
    mean_memory = ("memory_pct", "mean"),
    max_response = ("response_ms", "max"),
    total_rows = ("cpu_pct", "count")
).round(2).sort_values("mean_cpu", ascending=False)

,mean_cpu,max_cpu,mean_memory,max_response,total_rows
server_id,,,,,
srv-05,62.33,96.7,62.10,1196.4,105
srv-01,59.93,98.4,62.51,1196.2,105
srv-02,59.93,98.2,64.46,1196.1,105
srv-04,59.41,98.6,62.21,1159.8,105
srv-03,58.57,98.5,65.09,1195.8,105


In [21]:
# Task 2 — server_metrics
# Group by server_id AND status. Count rows per combination. 
# Which server has the most critical status readings?

In [22]:
# Group by server_id AND status. Count rows per combination. 
df_metrics.groupby(["server_id", "status"], observed=True)["cpu_pct"].count()

server_id  status  
srv-01     critical     5
           ok          79
           warning     21
srv-02     critical     7
           ok          84
           warning     14
srv-03     critical     7
           ok          85
           warning     13
srv-04     critical     4
           ok          77
           warning     24
srv-05     critical     7
           ok          79
           warning     19
Name: cpu_pct, dtype: int64

In [23]:
# Which server has the most critical status readings?
result = df_metrics.groupby(["server_id", "status"], observed=True)["cpu_pct"].count().reset_index(name="row_count")
print(result)

#Filter critical only and find the max
critical = result[result["status"] == "critical"].sort_values("row_count", ascending=False)
print("\nmax critical\n", critical)
# srv-02 & srv-03 are tied with 7 critical reading each.(max)

   server_id    status  row_count
0     srv-01  critical          5
1     srv-01        ok         79
2     srv-01   warning         21
3     srv-02  critical          7
4     srv-02        ok         84
5     srv-02   warning         14
6     srv-03  critical          7
7     srv-03        ok         85
8     srv-03   warning         13
9     srv-04  critical          4
10    srv-04        ok         77
11    srv-04   warning         24
12    srv-05  critical          7
13    srv-05        ok         79
14    srv-05   warning         19

max critical
    server_id    status  row_count
3     srv-02  critical          7
6     srv-03  critical          7
12    srv-05  critical          7
0     srv-01  critical          5
9     srv-04  critical          4


In [24]:
# Task 3 — incidents
# Group by category. Compute total tickets, resolved count, unresolved count, and 
# resolution rate %. Sort by resolution rate ascending — which category is hardest to resolve?

In [25]:
df_tickets.head(5)

,ticket_id,server_id,category,priority,created_at,resolved_at,resolved
0,INC0001,srv-04,Service Down,2,2026-03-24 02:00:00,NaT,False
1,INC0002,srv-01,CPU High,3,2026-03-19 15:00:00,2026-03-21 12:00:00,True
2,INC0003,srv-03,Service Down,2,2026-04-02 10:00:00,NaT,False
3,INC0004,srv-02,Service Down,1,2026-03-13 20:00:00,NaT,False
4,INC0005,srv-04,Disk Full,2,2026-03-18 09:00:00,2026-03-21 01:00:00,True


In [26]:
summary = df_tickets.groupby("category", observed=True).agg(
    total_tickets = ("ticket_id", "count"),
    # resolved_count = ("resolved", "count"), #true
    resolved_count = ("resolved", lambda x: (x==True).sum()),
    unresolved_count = ("resolved", lambda x: (x==False).sum()),
)
print(summary)

              total_tickets  resolved_count  unresolved_count
category                                                     
CPU High                 50              33                17
Disk Full                46              26                20
Memory Leak              36              22                14
Network Lag              35              19                16
Service Down             46              25                21


In [27]:
summary["resolution_rate"] = summary["resolved_count"] / summary["total_tickets"] * 100
print(summary)

              total_tickets  resolved_count  unresolved_count  resolution_rate
category                                                                      
CPU High                 50              33                17        66.000000
Disk Full                46              26                20        56.521739
Memory Leak              36              22                14        61.111111
Network Lag              35              19                16        54.285714
Service Down             46              25                21        54.347826


In [28]:
print(summary.sort_values("resolution_rate", ascending=True))

              total_tickets  resolved_count  unresolved_count  resolution_rate
category                                                                      
Network Lag              35              19                16        54.285714
Service Down             46              25                21        54.347826
Disk Full                46              26                20        56.521739
Memory Leak              36              22                14        61.111111
CPU High                 50              33                17        66.000000


In [29]:
# Using ASSIGN to solve the problem with less lines of code

In [30]:
summary = df_tickets.groupby("category").agg(
    total_tickets = ("ticket_id", "count"),
    resolved_count = ("resolved", lambda x: (x==True).sum()),
    unresolved_count = ("resolved", lambda x: (x==False).sum()),
).assign(
    resolution_rate = lambda x : (x["resolved_count"] / x["total_tickets"] * 100).round(1))
print(summary)

              total_tickets  resolved_count  unresolved_count  resolution_rate
category                                                                      
CPU High                 50              33                17             66.0
Disk Full                46              26                20             56.5
Memory Leak              36              22                14             61.1
Network Lag              35              19                16             54.3
Service Down             46              25                21             54.3


In [31]:
# CONCLUSION - which category is hardest to resolve?

# Hardest to resolved - Network Lag - 54.3% resolution rate (lowest)
# Easiest to resolve - CPU High - 66% resolution rate (highest)

# AIOps insight
# Network Lag and Service Down are infrastructure-level issues — harder to resolve because 
# they often depend on external factors (ISP, hardware, third-party services). 
# CPU High is easier because it's usually a process kill or autoscale trigger.

In [32]:
# Task 4 — incidents
# Group by server_id. Find mean priority per server. 
# Which server has the lowest mean priority number (meaning most P1 tickets)?

In [33]:
df_tickets.head(5)

,ticket_id,server_id,category,priority,created_at,resolved_at,resolved
0,INC0001,srv-04,Service Down,2,2026-03-24 02:00:00,NaT,False
1,INC0002,srv-01,CPU High,3,2026-03-19 15:00:00,2026-03-21 12:00:00,True
2,INC0003,srv-03,Service Down,2,2026-04-02 10:00:00,NaT,False
3,INC0004,srv-02,Service Down,1,2026-03-13 20:00:00,NaT,False
4,INC0005,srv-04,Disk Full,2,2026-03-18 09:00:00,2026-03-21 01:00:00,True


In [34]:
df_tickets.groupby(["server_id"])["priority"].mean().round(2).sort_values(ascending=True)

server_id
srv-01    1.87
srv-03    1.90
srv-04    1.91
srv-02    2.00
srv-05    2.00
Name: priority, dtype: float64

In [35]:
# Which server has the lowest mean priority number (meaning most P1 tickets)?
# srv-01 has the lowest mean priority number meaning most P1 tickets.

In [36]:
# Task 5 — app_logs
# Group by server_id and log_level. Count entries per combination. 
# Unstack to wide format. Add an error_rate column = (ERROR + CRITICAL) / total × 100.

In [37]:
df_metrics.dtypes

timestamp        datetime64[ns]
server_id              category
cpu_pct                 float64
memory_pct              float64
response_ms             float64
disk_pct                float64
status                   object
fleet_avg_cpu           float64
cpu_deviation           float64
dtype: object

In [38]:
df_tickets.dtypes

ticket_id              object
server_id              object
category               object
priority                int64
created_at     datetime64[ns]
resolved_at    datetime64[ns]
resolved                 bool
dtype: object

In [39]:
df_logs.dtypes

timestamp     datetime64[ns]
server_id             object
log_level             object
message               object
error_code            object
dtype: object

In [40]:
df_logs.groupby(["server_id", "log_level"]).count()

timestamp  message  error_code
server_id log_level                                
srv-01    CRITICAL           7        7           5
          ERROR              5        5           5
          INFO              26       26          18
          WARNING            7        7           7
srv-02    CRITICAL           8        8           6
          ERROR             14       14          11
          INFO              32       32          25
          WARNING           18       18          16
srv-03    CRITICAL           6        6           5
          ERROR             11       11           9
          INFO              30       30          26
          WARNING           19       19          18
srv-04    CRITICAL          10       10           9
          ERROR              6        6           6
          INFO              27       27          26
          WARNING           20       20          15
srv-05    CRITICAL          10       10           9
          ERROR             11       11           8
          INFO              34       34          30
          WARNING           14       14          10

In [41]:
df_logs.groupby(["server_id", "log_level"]).size()

server_id  log_level
srv-01     CRITICAL      7
           ERROR         5
           INFO         26
           WARNING       7
srv-02     CRITICAL      8
           ERROR        14
           INFO         32
           WARNING      18
srv-03     CRITICAL      6
           ERROR        11
           INFO         30
           WARNING      19
srv-04     CRITICAL     10
           ERROR         6
           INFO         27
           WARNING      20
srv-05     CRITICAL     10
           ERROR        11
           INFO         34
           WARNING      14
dtype: int64

In [42]:
# Key Difference (very important)
# Method	What it does
# count()	counts non-null values per column
# size()	counts total rows per group 

# For log analytics → always prefer size()

In [43]:
# Unstack to wide format. Add an error_rate column = (ERROR + CRITICAL) / total × 100.

In [44]:
counts = df_logs.groupby(["server_id", "log_level"]).size()

In [45]:
wide = counts.unstack(fill_value=0)

wide["total"] = wide.sum(axis=1)

wide["ERROR"] = wide.get("ERROR", 0)
wide["CRITICAL"] = wide.get("CRITICAL", 0)

wide["error_rate"] = (
    (wide["ERROR"] + wide["CRITICAL"]) / wide["total"] * 100
).round(2)
print(wide["error_rate"])

server_id
srv-01    26.67
srv-02    30.56
srv-03    25.76
srv-04    25.40
srv-05    30.43
Name: error_rate, dtype: float64


In [46]:
# TOP ERROR RATE SERVERS
print(wide["error_rate"].sort_values(ascending=False))
print(wide["error_rate"].sort_values(ascending=False).head(5))

server_id
srv-02    30.56
srv-05    30.43
srv-01    26.67
srv-03    25.76
srv-04    25.40
Name: error_rate, dtype: float64
server_id
srv-02    30.56
srv-05    30.43
srv-01    26.67
srv-03    25.76
srv-04    25.40
Name: error_rate, dtype: float64


In [47]:
# ALTERNATIVE WAY
log_summary = df_logs.groupby(
    ["server_id", "log_level"]
).size().unstack(fill_value=0).assign(
    total      = lambda x: x.sum(axis=1),
    error_rate = lambda x: ((x["ERROR"] + x["CRITICAL"]) / x["total"] * 100).round(1)
)

print(log_summary.sort_values(by="error_rate", ascending=False))

log_level  CRITICAL  ERROR  INFO  WARNING  total  error_rate
server_id                                                   
srv-02            8     14    32       18     72        30.6
srv-05           10     11    34       14     69        30.4
srv-01            7      5    26        7     45        26.7
srv-03            6     11    30       19     66        25.8
srv-04           10      6    27       20     63        25.4


In [48]:
# Task 6 — app_logs
# Group by error_code. Count how many times each error code appears. 
# Which error code is most frequent? Exclude None / null error codes from the count.

In [49]:
df = df_logs[df_logs["error_code"].notna()] # Exclude null error codes
counts = df["error_code"].value_counts()
print(counts)

error_code
E001    78
E003    72
E002    66
E004    48
Name: count, dtype: int64


In [50]:
most_frequent = counts.idxmax()
print("Most frequent error code:", most_frequent)

Most frequent error code: E001


In [51]:
# Alternative way 
counts = (
    df_logs[df_logs["error_code"].notna()]
    .groupby("error_code")
    .size()
    .sort_values(ascending=False)
)

if not counts.empty:
    print("Most frequent:", counts)  # or print("Most frequent:", counts.index[0])
else:
    print("No valid error codes")

Most frequent: error_code
E001    78
E003    72
E002    66
E004    48
dtype: int64


In [52]:
# Another way
most_frequent = (
    df_logs["error_code"]
    .dropna()
    .value_counts()
    .idxmax()
)
print(most_frequent)

E001


In [53]:
# Task 7 — cross-dataset
# Use transform() to add server_avg_cpu to df_metrics. 
# Then flag rows where cpu_pct > server_avg_cpu * 1.3 as above_threshold. 
# Join the flagged metrics with df_tickets on server_id. 
# How many above-threshold metric readings have an open P1 ticket on the same server?

In [54]:
df_metrics["server_avg_cpu"] = (
    df_metrics.groupby("server_id",observed=True)["cpu_pct"].transform("mean")
)

df_metrics["above_threshold"] = (
    df_metrics["cpu_pct"] > df_metrics["server_avg_cpu"] * 1.3
)

flagged = df_metrics[df_metrics["above_threshold"]]

p1_open = df_tickets[
    (df_tickets["priority"] == 1) &
    (df_tickets["resolved"] == False)
    ]

p1_servers = p1_open["server_id"].unique()

count = flagged[flagged["server_id"].isin(p1_servers)].shape[0]

print("Above-threshold readings with open P1 tickets:", count)

Above-threshold readings with open P1 tickets: 150
